# Recommendation system on MovieLens dataset

We build a recommendation system on the MovieLens dataset. The system is a two-stage retrieval-ranker system, consisting of an ALS matrix factorisation model to retrieval relevant items for a given user, followed by an XGBoost ranker model which ranks the retrieved items and recommends a subset of those as highest ranked items.

In this notebookd, we use the MovieLens dataset. The task is to recommend k movies for each user. We demonstrate the following crucial aspects:

- Methodology
- Train-validation(s) split strategy
- Evaluation strategy


## Methodology

![retrieval_ranker](retrieval_ranker.jpg)

Retrieval-ranker system: ALS matrix factorisation followed by an XGBoost ranker:

- fit an ALS on 'train_als' set. Evaluate recall@N of the ALS (maximise recall@N such that ALS recommends more of the items in the validation set ('val_als'))
- Use fitted ALS to generate N candidate items
- Use 'val_als' historical user-items to label the N candidate items
- Include additional features e.g. movie genre and user activity
- The resulting labelled dataset is 'train_ranker', which has used both 'train_als' and 'val_als'
- Fit xgboost ranker. Number of recommendation is k (defaults to 5 but no hardcoding) and evaluate on a validation set that ALS has never ever seen (not even the 'val_als' set); call it 'val_ranker': evaluate precision@k, recall@k, and NDCG@k


### Train-validation(s) split

The two-stage system requires the following split:

- 'train_als': ALS training data
- 'val_als' validation data
- 'train_ranker': training set for ranker, which has used both 'train_als' and 'val_als'
- 'val_ranker': never ever seen by ALS and ranker during their trainings

To mimick production settings, we adopt a global temporal split. A common alternative: i.e. leave most recent X items per user, is disadvantageous as the recency of users are different.

1. From the dataset, get the time range of the records.
2. Perform a global temporal split with the following ratios
    - 80% 'train_als'
    - 20% 'val_als'
    - 20% 'val_ranker'
3. Trained ALS on 'train_als'
4. Make 'train_ranker': generate candidates from ALS, make labels from 'val_als'
5. Make 'val_ranker': generate candidates from ALS, make labels from 'val_ranker'

This provides a clean separation and prevents data leakage between ALS and ranker training and validation.

### Retrieval model: Alternating Least Square (ALS) rating matrix factorisation

![Rating Matrix](rating_matrix.png)

For the retrieval model, we use ALS to factorise the user-movie rating matrix into [user matrix] x [movie matrix] over a latent space.

In the two-stage system, the retrieval model needs to generate as much relevant candidate items (those which shows up in 'val_als') as possible, so that they can enter the ranker system and the final k items. I.e. we maximise `Recall@N`.

ALS minimises reconstruction error on the populated matrix elements; this is different from the recommendation objectives so often hyperparameter tuning is required. 


### XGBoost ranker model

The job of the second-stage ranker model is to rank the N candidates coming out of the ALS retrieval model, and select the top k relevant items. The objectives are then to maximise

- `Precision@k`
- `NDCG@k`


### Metrics

- Precision@K: number of movies hits (relevant movies) out of K recommended
- Recall@K: number of movies hits out of K user actually watched
- NDCG@K: takes into account the order in which the recommendation is made--- we want the relevant recs to show up earlier
    - Discounted Cumulative Gain (DCG) = sum_{i=1}^K [relevance_score] / log_2(i+1)
    - Ideal DCG: DCG when the relevant movies hit first
    - NDCG: DCG/IDCG
    - Quantifies how close a model's ranking is to the ideal order (0 is worst, 1 is best)


In [1]:
import os
from copy import deepcopy

import numpy as np

# import logging
import pandas as pd
from implicit.als import AlternatingLeastSquares
from scipy.sparse import csr_matrix

os.environ["OPENBLAS_NUM_THREADS"] = "1"  # ALS complains
import matplotlib.pyplot as plt
from sklearn.metrics import ndcg_score
from xgboost import XGBRanker

from helper_funcs import (
    evaluate_als_model,
    evaluate_ranker_model,
    make_movie_features,
    make_ranker_labelled_dataset,
    make_user_features,
    recommend_candidates,
    recommend_for_user,
)
from utils import (
    load_dataset,
    load_yaml,
)

# logger = logging.getLogger(__name__)
# logger.setLevel(logging.INFO)
# logger.addHandler(logging.StreamHandler())

RANDOM_SEED = 42

rng = np.random.RandomState(RANDOM_SEED)

%load_ext autoreload
%autoreload 2

/home/cedricyu/projects/recommendation_system/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Config and dataset, make ID mappings

In [2]:
config_path = "config.yml"
config = load_yaml(config_path)

N_CANDIDATES = config["retrieval_specs"]["n_candidates"]
K = config["ranker_specs"]["k"]

In [3]:
ratings, movies = load_dataset(config["data_dir"])
# ratings is sorted by timestamp in ascending order for temporal splitting

In [4]:
import gc
gc.collect()

0

In [5]:
all_users = np.sort(ratings.userId.unique())
all_movies = np.sort(ratings.movieId.unique())

user_to_idx = {u: i for i, u in enumerate(all_users)}
idx_to_user = {i: u for u, i in user_to_idx.items()}

movie_to_idx = {m: i for i, m in enumerate(all_movies)}
idx_to_movie = {i: m for m, i in movie_to_idx.items()}

In [6]:
ratings.head()

,userId,movieId,rating,timestamp
15254009,149954,1176,4.0,789652004
9617119,94532,47,5.0,789652009
9617113,94532,21,3.0,789652009
9617162,94532,1079,3.0,789652009
16958936,166476,52,4.0,822873600


In [7]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [8]:
(
    ratings["userId"].nunique(),
    ratings["movieId"].nunique(),
    ratings.groupby("userId")["movieId"].count().median(),
)

(330975, 83239, np.float64(31.0))

## Train-validation(s) split

In [9]:
split_specs = config["split_specs"]
split_specs

{'train_als_frac': 0.8, 'val_als_frac': 0.1, 'val_ranker_frac': 0.1}

In [10]:
train_end = int(len(ratings) * split_specs["train_als_frac"])
val_als_end = int(
    len(ratings) * (split_specs["train_als_frac"] + split_specs["val_als_frac"])
)

train_als = ratings.iloc[:train_end].sort_values(["userId", "timestamp"]).copy()
val_als = (
    ratings.iloc[train_end:val_als_end].sort_values(["userId", "timestamp"]).copy()
)
val_ranker = ratings.iloc[val_als_end:].sort_values(["userId", "timestamp"]).copy()


train_als_user_items = train_als.groupby("userId")["movieId"].apply(list).to_dict()

val_als_user_items = val_als.groupby("userId")["movieId"].apply(list).to_dict()

val_ranker_user_items = val_ranker.groupby("userId")["movieId"].apply(list).to_dict()


print("\nSplit sizes")
print(f"train_als  : {len(train_als):,}")
print(f"val_als    : {len(val_als):,}")
print(f"val_ranker : {len(val_ranker):,}")

print("\nSplit time ranges")
print(
    f"train_als  : {ratings['timestamp'].iloc[:train_end].min()} to {ratings['timestamp'].iloc[:train_end].max()}"
)
print(
    f"val_als    : {ratings['timestamp'].iloc[train_end:val_als_end].min()} to {ratings['timestamp'].iloc[train_end:val_als_end].max()}"
)
print(
    f"val_ranker : {ratings['timestamp'].iloc[val_als_end:].min()} to {ratings['timestamp'].iloc[val_als_end:].max()}"
)


Split sizes
train_als  : 27,065,729
val_als    : 3,383,216
val_ranker : 3,383,217

Split time ranges
train_als  : 789652004 to 1531094315
val_als    : 1531094329 to 1598691955
val_ranker : 1598691989 to 1689843213


In [11]:
(
    train_als["userId"].nunique(),
    train_als["movieId"].nunique(),
    train_als.groupby("userId")["movieId"].count().median(),
)

(280422, 49383, np.float64(30.0))

## Retrieval model

ALS: make `train_als` sparse matrix, fit ALS and evaluate on `val_als`

In [12]:
# make user-item sparse matrix

# map userId and movieId to integer indices
train_als["user_idx"] = train_als.userId.map(user_to_idx)
train_als["movie_idx"] = train_als.movieId.map(movie_to_idx)

user_item_matrix = csr_matrix(
    (train_als["rating"], (train_als["user_idx"], train_als["movie_idx"]))
)

user_item_matrix.shape

(330975, 53346)

In [13]:
als_model = AlternatingLeastSquares(
    **config["retrieval_model_specs"], random_state=RANDOM_SEED
)
als_model.fit(user_item_matrix)

print(f"Fitted ALS model")

/home/cedricyu/projects/recommendation_system/.venv/lib/python3.10/site-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 16 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
100%|██████████| 20/20 [01:18<00:00,  3.95s/it]

Fitted ALS model


### Evaluate ALS on `val_als`

In [14]:
val_als_recall = evaluate_als_model(
    als_model=als_model,
    train_als_user_items=train_als_user_items,
    val_als_user_items=val_als_user_items,
    user_item_matrix=user_item_matrix,
    user_to_idx=user_to_idx,
    idx_to_movie=idx_to_movie,
    n_candidates=N_CANDIDATES,
)

In [15]:
print(f"Retrieval model: Recall@{N_CANDIDATES} = {val_als_recall:.4f}")

Retrieval model: Recall@100 = 0.1846


## Second-stage ranking model: make features and label

### Movie features: onehot-encoded genre and popularity (on `train_als`)

In [16]:
movie_features = make_movie_features(movies=movies, train_als=train_als)

In [17]:
movie_features.head()

,movieId,genre_Action,genre_Adventure,genre_Animation,genre_Children,genre_Comedy,genre_Crime,genre_Documentary,genre_Drama,genre_Fantasy,...,genre_Horror,genre_IMAX,genre_Musical,genre_Mystery,genre_Romance,genre_Sci-Fi,genre_Thriller,genre_War,genre_Western,movie_popularity
0,1,0,1,1,1,1,0,0,0,1,...,0,0,0,0,0,0,0,0,0,67739.0
1,2,0,1,0,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,26788.0
2,3,0,0,0,0,1,0,0,0,0,...,0,0,0,0,1,0,0,0,0,15547.0
3,4,0,0,0,0,1,0,0,1,0,...,0,0,0,0,1,0,0,0,0,2986.0
4,5,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,15407.0


### User features: user movie view count and mean rating in `train_als`

In [18]:
user_features = make_user_features(train_als=train_als)

In [19]:
user_features.head()

,userId,user_activity,user_avg_rating
0,1,62,4.008065
1,2,91,3.527473
2,4,30,4.366667
3,5,43,3.418605
4,6,48,3.968750


### Making ranker training and validation sets

In [20]:
train_ranker = make_ranker_labelled_dataset(
    als_model=als_model,
    train_als_user_items=train_als_user_items,
    validation_user_items=val_als_user_items,
    user_item_matrix=user_item_matrix,
    user_to_idx=user_to_idx,
    idx_to_movie=idx_to_movie,
    n_candidates=N_CANDIDATES,
    user_features=user_features,
    movie_features=movie_features,
)

In [21]:
train_ranker

,userId,movieId,als_score,als_rank,label,user_activity,user_avg_rating,genre_Action,genre_Adventure,genre_Animation,...,genre_Horror,genre_IMAX,genre_Musical,genre_Mystery,genre_Romance,genre_Sci-Fi,genre_Thriller,genre_War,genre_Western,movie_popularity
0,95,166528,1.010162,0,0,329,3.267477,1,1,0,...,0,0,0,0,0,1,0,0,0,5055.0
1,95,168252,1.005871,1,0,329,3.267477,1,0,0,...,0,0,0,0,0,1,0,0,0,4628.0
2,95,60069,0.985205,2,0,329,3.267477,0,1,1,...,0,0,0,0,1,1,0,0,0,27161.0
3,95,3000,0.916054,3,0,329,3.267477,1,1,1,...,0,0,0,0,0,0,0,0,0,13549.0
4,95,122886,0.915131,4,0,329,3.267477,1,1,0,...,0,1,0,0,0,1,0,0,0,12047.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
696895,330942,2001,0.218829,95,0,47,3.531915,1,0,0,...,0,0,0,0,0,0,0,0,0,15839.0
696896,330942,1729,0.216638,96,0,47,3.531915,0,0,0,...,0,0,0,0,0,0,1,0,0,13878.0
696897,330942,134130,0.216354,97,0,47,3.531915,0,1,0,...,0,0,0,0,0,1,0,0,0,15250.0
696898,330942,1645,0.214321,98,0,47,3.531915,0,0,0,...,0,0,0,1,0,0,1,0,0,14970.0


In [22]:
# label imbalance: aim at around 5%
train_ranker["label"].mean()

np.float64(0.09845458458889367)

In [23]:
val_ranker = make_ranker_labelled_dataset(
    als_model=als_model,
    train_als_user_items=train_als_user_items,
    validation_user_items=val_ranker_user_items,
    user_item_matrix=user_item_matrix,
    user_to_idx=user_to_idx,
    idx_to_movie=idx_to_movie,
    n_candidates=N_CANDIDATES,
    user_features=user_features,
    movie_features=movie_features,
)

In [24]:
val_ranker["label"].mean()

np.float64(0.07786610878661088)

In [25]:
val_ranker

,userId,movieId,als_score,als_rank,label,user_activity,user_avg_rating,genre_Action,genre_Adventure,genre_Animation,...,genre_Horror,genre_IMAX,genre_Musical,genre_Mystery,genre_Romance,genre_Sci-Fi,genre_Thriller,genre_War,genre_Western,movie_popularity
0,137,48780,1.284196,0,0,363,4.190083,0,0,0,...,0,0,0,1,0,1,1,0,0,22353.0
1,137,54503,1.202295,1,0,363,4.190083,0,0,0,...,0,0,0,0,0,0,0,0,0,12497.0
2,137,79132,1.071279,2,0,363,4.190083,1,0,0,...,0,1,0,1,0,1,1,0,0,39838.0
3,137,148626,1.058529,3,0,363,4.190083,0,0,0,...,0,0,0,0,0,0,0,0,0,6310.0
4,137,47610,1.033520,4,0,363,4.190083,0,0,0,...,0,0,0,1,1,0,0,0,0,11933.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
477995,330914,5574,0.781270,95,1,783,3.659642,1,0,0,...,0,0,0,0,0,0,0,0,0,5885.0
477996,330914,2028,0.780364,96,1,783,3.659642,1,0,0,...,0,0,0,0,0,0,0,1,0,53172.0
477997,330914,85510,0.778773,97,0,783,3.659642,1,0,0,...,0,1,0,0,0,0,1,0,0,2345.0
477998,330914,79134,0.776734,98,1,783,3.659642,0,0,0,...,0,0,0,0,0,0,0,0,0,1574.0


## Train ranker model

In [26]:
exclude_cols = {
    "userId",
    "movieId",
    "label",
}

feature_cols = [c for c in train_ranker.columns if c not in exclude_cols]

X_train = train_ranker[feature_cols]
y_train = train_ranker["label"]

group_train = train_ranker.groupby("userId").size().values

ranker_model = XGBRanker(**config["ranker_model_specs"], random_state=RANDOM_SEED)

ranker_model.fit(
    X_train,
    y_train,
    group=group_train,
)

XGBRanker(base_score=None, booster=None, callbacks=None, colsample_bylevel=None,
          colsample_bynode=None, colsample_bytree=0.8, device=None,
          early_stopping_rounds=None, enable_categorical=False,
          eval_metric=None, feature_types=None, feature_weights=None,
          gamma=None, grow_policy=None, importance_type=None,
          interaction_constraints=None, learning_rate=0.05, max_bin=None,
          max_cat_threshold=None, max_cat_to_onehot=None, max_delta_step=None,
          max_depth=6, max_leaves=None, min_child_weight=None, missing=nan,
          monotone_constraints=None, multi_strategy=None, n_estimators=300,
          n_jobs=None, num_parallel_tree=None, ...)

### Evaluate ranker model on `val_ranker`

In [27]:
ranker_precision, ranker_recall, ranker_ndcg = evaluate_ranker_model(
    ranker_model=ranker_model, val_ranker=val_ranker, feature_cols=feature_cols, k=K
)


Final metrics
Precision@10: 0.1716
Recall@10: 0.2044
NDCG@10: 0.2283


## Inference: recommend top-K for user

In [28]:
user_id = 1

recommendations = recommend_for_user(
    user_id=1,
    als_model=als_model,
    ranker_model=ranker_model,
    user_item_matrix=user_item_matrix,
    user_to_idx=user_to_idx,
    idx_to_movie=idx_to_movie,
    feature_cols=feature_cols,
    n_candidates=N_CANDIDATES,
    k=K,
    user_features=user_features,
    movie_features=movie_features,
)

In [ ]:
recommendations

,userId,movieId,als_score,als_rank,user_activity,user_avg_rating,genre_Action,genre_Adventure,genre_Animation,genre_Children,...,genre_IMAX,genre_Musical,genre_Mystery,genre_Romance,genre_Sci-Fi,genre_Thriller,genre_War,genre_Western,movie_popularity,ranker_score
96,1,318,0.254607,96,62,4.008065,0,0,0,0,...,0,0,0,0,0,0,0,0,96216.0,0.511188
3,1,1240,0.640751,3,62,4.008065,1,0,0,0,...,0,0,0,0,1,1,0,0,41638.0,0.425176
1,1,6377,0.765554,1,62,4.008065,0,1,1,1,...,0,0,0,0,0,0,0,0,36123.0,0.364595
0,1,589,0.770932,0,62,4.008065,1,0,0,0,...,0,0,0,0,1,0,0,0,63560.0,0.319608
2,1,1198,0.754836,2,62,4.008065,1,1,0,0,...,0,0,0,0,0,0,0,0,62484.0,0.115406
6,1,8368,0.553890,6,62,4.008065,0,1,0,0,...,1,0,0,0,0,0,0,0,22728.0,0.080358
4,1,1270,0.621349,4,62,4.008065,0,1,0,0,...,0,0,0,0,1,0,0,0,56716.0,0.072835
7,1,3147,0.543533,7,62,4.008065,0,0,0,0,...,0,0,0,0,0,0,0,0,32432.0,0.072569
25,1,5989,0.394214,25,62,4.008065,0,0,0,0,...,0,0,0,0,0,0,0,0,28030.0,0.037641
19,1,1197,0.445192,19,62,4.008065,1,1,0,0,...,0,0,0,1,0,0,0,0,42285.0,0.024723


: 